In [2]:
import json
import os
import spacy
from spacy.tokens import DocBin
from tqdm import tqdm
from spacy.util import filter_spans

# Dossier contenant tous les fichiers JSON
DOSSIER_JSON = "JSON entraînementY"

# Liste des classes à adapter à ton dataset
CLASSES = ['PER', 'LOC', 'ORG', 'MISC']  # adapte-les si besoin

# Initialisation du modèle spaCy
nlp = spacy.blank("fr")

# Parcours de tous les fichiers JSON du dossier
for nom_fichier in os.listdir(DOSSIER_JSON):
    if nom_fichier.endswith(".json"):
        chemin_fichier = os.path.join(DOSSIER_JSON, nom_fichier)
        print(f"\n📄 Traitement de : {nom_fichier}")

        # Charger le fichier JSON
        with open(chemin_fichier, 'r', encoding='utf-8') as f:
            data = json.load(f)

        training_data = {'classes': CLASSES, 'annotations': []}

        # Construction du training_data
        for example in data['examples']:
            temp_dict = {
                'text': example['content'],
                'entities': []
            }

            for annotation in example['annotations']:
                start = annotation['start']
                end = annotation['end']
                label = annotation['tag_name'].upper()
                temp_dict['entities'].append((start, end, label))

            training_data['annotations'].append(temp_dict)

        # Conversion en format spaCy
        doc_bin = DocBin()
        for training_example in tqdm(training_data['annotations'], desc=f"Conversion {nom_fichier}"):
            text = training_example['text']
            labels = training_example['entities']
            doc = nlp.make_doc(text)
            ents = []

            for start, end, label in labels:
                span = doc.char_span(start, end, label=label, alignment_mode="expand")
                if span is None:
                    print(f"⚠️ Skipping entity: ({start}, {end}, {label})")
                else:
                    ents.append(span)

            doc.ents = filter_spans(ents)
            doc_bin.add(doc)

        # Nom du fichier de sortie .spacy
        nom_spacy = nom_fichier.replace(".json", ".spacy")
        chemin_spacy = os.path.join(DOSSIER_JSON, nom_spacy)
        doc_bin.to_disk(chemin_spacy)
        print(f"✅ Fichier .spacy créé : {nom_spacy}")


📄 Traitement de : ensemble_complet.json


Conversion ensemble_complet.json: 100%|███████████| 6/6 [00:00<00:00, 18.09it/s]


✅ Fichier .spacy créé : ensemble_complet.spacy

📄 Traitement de : ensemble_25pourcent.json


Conversion ensemble_25pourcent.json: 100%|████████| 6/6 [00:00<00:00, 20.16it/s]


✅ Fichier .spacy créé : ensemble_25pourcent.spacy

📄 Traitement de : ensemble_REF.json


...

✅ Fichier .spacy créé : ensemble_Kraken.spacy

📄 Traitement de : ensemble_75pourcent.json


Conversion ensemble_75pourcent.json: 100%|████████| 6/6 [00:00<00:00, 20.05it/s]


✅ Fichier .spacy créé : ensemble_75pourcent.spacy

📄 Traitement de : ensemble_50pourcent.json


Conversion ensemble_50pourcent.json: 100%|████████| 6/6 [00:00<00:00, 20.29it/s]

✅ Fichier .spacy créé : ensemble_50pourcent.spacy
